# Pulkkinen (2007) — Space Weather: Terrestrial Perspective — Implementation / 구현

**Paper**: Pulkkinen, T. (2007), *Space Weather: Terrestrial Perspective*, Living Reviews in Solar Physics, 4, 1. doi:10.12942/lrsp-2007-1.

This notebook implements and visualizes the 9 key mathematical frameworks referenced in the review, with all 8 planned modules:

1. **Akasofu ε-parameter** — Halloween 2003 time series reconstruction
2. **Poynting flux focusing** — conceptual visualization of §5 energy entry
3. **Harris current sheet** — substorm growth phase thinning
4. **Volland-Stern + Boyle** — magnetospheric convection potential maps
5. **Guiding-center drift** — ring current particle trajectories
6. **GIC induction** — 1-D conductivity model, surface E-field spectrum
7. **Dessler-Parker-Sckopke** — Dst ↔ ring current energy
8. **Radial diffusion** — relativistic electron L-shell evolution
9. **Summary** — connections to operational forecasting and later papers

이 노트북은 Pulkkinen 리뷰에서 참조하는 9개 핵심 수학 프레임워크를 구현/시각화합니다. 데이터는 합성(synthetic)이며 논문 값에 맞춰 보정됩니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime, timedelta
from scipy.integrate import odeint, solve_ivp

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
np.random.seed(42)

# Physical constants
R_E = 6.371e6          # Earth radius (m)
mu0 = 4 * np.pi * 1e-7 # Vacuum permeability (T m/A)
e_charge = 1.602e-19   # Elementary charge (C)
m_p = 1.673e-27        # Proton mass (kg)
m_e = 9.109e-31        # Electron mass (kg)
B0_Earth = 31_000e-9   # Earth's equatorial surface field (T)

## Part 1: Akasofu ε-Parameter / Akasofu ε 파라미터

**English**: The empirical solar-wind/magnetosphere coupling function (§5, Eq. 3):

$$\varepsilon = v B^2 \sin^4(\theta_c/2)\, L_0^2, \quad L_0 = 7\, R_E$$

where $\theta_c = \tan^{-1}(B_y/B_z)$ is the IMF clock angle. The function proxies the Poynting flux entering the magnetosphere but **breaks down after IMF turns northward** (Figure 12 of the paper).

**Korean**: 태양풍-자기권 결합 함수. IMF $B_z$가 남향일 때 최대; 북향 전환 후 0으로 떨어지지만 실제 MHD는 에너지 진입이 지속됨을 보임.

In [ ]:
def akasofu_epsilon(v_sw_kms: np.ndarray, Bz_nT: np.ndarray,
                    By_nT: np.ndarray, L0_RE: float = 7.0) -> np.ndarray:
    """Compute the Akasofu epsilon coupling function.

    Args:
        v_sw_kms: Solar wind speed (km/s).
        Bz_nT: GSM Bz component (nT).
        By_nT: GSM By component (nT).
        L0_RE: Empirical scaling length (R_Earth units).

    Returns:
        Epsilon in SI units (W).
    """
    v = v_sw_kms * 1e3  # m/s
    Bz = Bz_nT * 1e-9
    By = By_nT * 1e-9
    B_YZ = np.sqrt(By**2 + Bz**2)
    theta_c = np.arctan2(By, -Bz)
    theta_c = np.where(theta_c < 0, theta_c + 2 * np.pi, theta_c)
    theta_c_folded = np.where(theta_c > np.pi, 2 * np.pi - theta_c, theta_c)
    L0 = L0_RE * R_E
    eps = v * B_YZ**2 * np.sin(theta_c_folded / 2)**4 * L0**2 / mu0
    return eps


hours = np.arange(0, 96)
t = hours

v_sw = 400 + 50 * np.random.randn(96)
for t0, amp, tau in [(20, 1700, 6), (44, 900, 8), (72, 700, 10)]:
    v_sw += np.where(t >= t0, amp * np.exp(-(t - t0)/tau), 0)
v_sw = np.clip(v_sw, 300, 2500)

Bz = np.zeros(96)
for t0, amp, sign, tau in [(20, -30, 1, 8), (48, 20, -1, 6),
                            (60, -25, 1, 10), (80, 10, -1, 4)]:
    Bz += np.where(t >= t0, sign * amp * np.exp(-(t - t0)/tau), 0)
Bz += 2 * np.random.randn(96)

By = 5 * np.random.randn(96)

epsilon = akasofu_epsilon(v_sw, Bz, By)

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
axes[0].plot(t, v_sw, 'k')
axes[0].set_ylabel('v$_{sw}$ (km/s)')
axes[0].grid(alpha=0.3)

axes[1].plot(t, Bz, 'b', label='B$_z$')
axes[1].plot(t, By, 'g', alpha=0.6, label='B$_y$')
axes[1].axhline(0, ls='--', color='gray')
axes[1].set_ylabel('IMF (nT)')
axes[1].legend()
axes[1].grid(alpha=0.3)

axes[2].semilogy(t, np.clip(epsilon, 1e9, None), 'r')
axes[2].set_ylabel('ε (W)')
axes[2].set_xlabel('Hours from 28 Oct 2003 00 UT')
axes[2].grid(alpha=0.3, which='both')
axes[2].set_title('Note how ε drops whenever Bz turns north — but the paper argues this misses continuing energy entry')

plt.tight_layout()
plt.show()

print(f"Peak ε during storm: {epsilon.max():.2e} W")
print(f"Integrated energy entry (trapezoidal): {np.trapezoid(epsilon, t*3600):.2e} J")

## Part 2: Poynting Flux Focusing / Poynting flux 집중

**English**: §5 and Figure 10 show that the Poynting flux $\mathbf{S} = \mathbf{E}\times\mathbf{B}/\mu_0$ entering through the high-latitude magnetopause **focuses toward the inner magnetotail current sheet**. We visualize this conceptually with simple dipole+sheet geometry and streamlines.

**Korean**: Poynting flux는 고위도 magnetopause로 진입 후 꼬리 내부 current sheet로 집중되는 것을 간단한 쌍극자+시트 모델로 시각화.

In [ ]:
def magnetotail_field(x_RE: np.ndarray, z_RE: np.ndarray,
                      B_dip_nT: float = 30_000.0,
                      B_lobe_nT: float = 20.0,
                      lam_RE: float = 1.0) -> tuple:
    """Simple dipole + Harris tail superposition in noon-midnight plane.

    Args:
        x_RE: X-GSM (R_E), positive toward Sun.
        z_RE: Z-GSM (R_E).
        B_dip_nT: Surface dipole field (nT).
        B_lobe_nT: Tail lobe field (nT).
        lam_RE: Current sheet scale thickness (R_E).

    Returns:
        (Bx, Bz) in nT.
    """
    r2 = x_RE**2 + z_RE**2
    # Dipole (2D form)
    Bx_dip = -B_dip_nT * 3 * x_RE * z_RE / (r2**2.5 + 1e-6)
    Bz_dip = B_dip_nT * (2 * z_RE**2 - x_RE**2) / (r2**2.5 + 1e-6)
    # Harris tail for x < 0
    mask_tail = x_RE < -5
    Bx_tail = np.where(mask_tail, B_lobe_nT * np.tanh(z_RE / lam_RE), 0.0)
    Bz_tail = np.zeros_like(x_RE)
    return Bx_dip + Bx_tail, Bz_dip + Bz_tail


# Grid in noon-midnight plane
x = np.linspace(-40, 15, 120)
z = np.linspace(-15, 15, 80)
X, Z = np.meshgrid(x, z)

Bx, Bz = magnetotail_field(X, Z)
B_mag = np.sqrt(Bx**2 + Bz**2)

# Poynting flux streamlines: schematic energy flow based on Pulkkinen Fig. 10
# Energy enters high-latitude magnetopause and focuses to inner plasma sheet
def poynting_schematic(x, z):
    """Schematic Poynting flux pointing toward inner tail plasma sheet."""
    target_x, target_z = -12, 0
    dx = target_x - x
    dz = target_z - z
    # Only flows from outer regions inward toward target
    norm = np.sqrt(dx**2 + dz**2) + 1e-3
    Sx = dx / norm
    Sz = dz / norm
    # Only inside magnetosphere (simple parabolic boundary)
    mp_boundary = 10 - x / 3  # x = 10 R_E at subsolar, curves tailward
    inside = (np.abs(z) < mp_boundary) & (x > -40) & (x < 12)
    Sx = np.where(inside, Sx, np.nan)
    Sz = np.where(inside, Sz, np.nan)
    return Sx, Sz

Sx, Sz = poynting_schematic(X, Z)

fig, ax = plt.subplots(figsize=(13, 6))
ax.contourf(X, Z, np.log10(B_mag + 1), levels=30, cmap='viridis', alpha=0.4)
ax.streamplot(X, Z, Bx, Bz, color='white', density=1.2, linewidth=0.6,
              arrowsize=0.8)
ax.streamplot(X, Z, Sx, Sz, color='red', density=1.5, linewidth=1.5,
              arrowsize=1.5)

# Earth and magnetopause
theta = np.linspace(0, 2*np.pi, 200)
ax.fill(np.cos(theta), np.sin(theta), color='lightblue', zorder=5)
ax.plot(np.cos(theta), np.sin(theta), 'k', zorder=6)
# Plasma sheet
ax.axhline(0, xmin=0, xmax=0.65, color='orange', lw=3, alpha=0.7, zorder=3)
ax.text(-30, 0.8, 'Plasma sheet', color='orange', fontsize=11, ha='center')
ax.text(8, 5, 'Sun →', fontsize=11)
ax.text(-38, 13, 'Magnetotail lobe', fontsize=10)
ax.text(-35, -14, 'Magnetotail lobe', fontsize=10)

ax.set_xlim(-40, 15)
ax.set_ylim(-15, 15)
ax.set_aspect('equal')
ax.set_xlabel('X-GSM (R$_E$)')
ax.set_ylabel('Z-GSM (R$_E$)')
ax.set_title('Poynting flux (red) focusing toward inner plasma sheet — schematic of Fig. 10')
plt.tight_layout()
plt.show()

## Part 3: Harris Current Sheet and Substorm Thinning / Harris 전류 층과 substorm 얇아짐

**English**: The quiet magnetotail is approximated by the Harris profile (§6):

$$B_x(Z) = B_0 \tanh(Z/\lambda)$$

with current density $j_y = -(1/\mu_0)\,dB_x/dZ = -B_0/(\mu_0\lambda)\,\text{sech}^2(Z/\lambda)$. During substorm growth phase, the current sheet thins from $\lambda \sim 1\,R_E$ down to the **ion gyroradius** scale ($\lambda \sim 0.01\,R_E$), intensifying the current density by 100x.

**Korean**: 자기권 꼬리 정온 상태는 Harris profile로 근사. Substorm growth phase에서 current sheet가 ~$R_E$ 규모에서 ion gyroradius ($\sim 0.01 R_E$)까지 얇아져 전류 밀도가 100배 증가.

In [ ]:
def harris_profile(z_RE: np.ndarray, B0_nT: float,
                   lam_RE: float) -> tuple:
    """Harris current sheet profile and current density.

    Args:
        z_RE: Coordinate across sheet (R_E).
        B0_nT: Lobe magnetic field (nT).
        lam_RE: Sheet half-thickness (R_E).

    Returns:
        (B_x in nT, j_y in nA/m^2).
    """
    B0 = B0_nT * 1e-9
    lam = lam_RE * R_E
    z = z_RE * R_E
    Bx = B0 * np.tanh(z / lam)
    # j_y = (1/mu_0) dBx/dz
    jy = (B0 / (mu0 * lam)) * (1.0 - np.tanh(z / lam)**2)
    return Bx * 1e9, jy * 1e9  # Convert back: Bx in nT, jy in nA/m^2


z_grid = np.linspace(-3, 3, 600)
scenarios = [
    ("Quiet tail (λ = 1 R_E)",         20, 1.0,   '#4477AA'),
    ("Growth phase (λ = 0.3 R_E)",     30, 0.3,   '#EE7733'),
    ("Late growth (λ = 0.1 R_E)",      45, 0.1,   '#CC3311'),
    ("Near onset (λ = 0.02 R_E)",      60, 0.02,  '#000000'),
]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for label, B0, lam, color in scenarios:
    Bx, jy = harris_profile(z_grid, B0, lam)
    axes[0].plot(Bx, z_grid, label=label, color=color, lw=2)
    axes[1].plot(jy, z_grid, label=label, color=color, lw=2)

axes[0].set_xlabel('B$_x$ (nT)')
axes[0].set_ylabel('Z (R$_E$)')
axes[0].set_title('Magnetic field profile')
axes[0].grid(alpha=0.3)
axes[0].legend(fontsize=9)
axes[0].axhline(0, ls=':', color='gray')

axes[1].set_xlabel('Current density j$_y$ (nA/m$^2$)')
axes[1].set_ylabel('Z (R$_E$)')
axes[1].set_title('Cross-tail current density — note 1000x intensification at λ = 0.02 R$_E$')
axes[1].grid(alpha=0.3)
axes[1].legend(fontsize=9)
axes[1].set_xscale('symlog', linthresh=0.1)
axes[1].axhline(0, ls=':', color='gray')

plt.tight_layout()
plt.show()

# Peak current density comparison
for label, B0, lam, _ in scenarios:
    _, jy = harris_profile(np.array([0.0]), B0, lam)
    print(f"{label}: peak j_y = {jy[0]:.2f} nA/m^2")

## Part 4: Volland-Stern + Boyle Convection Potential / Volland-Stern + Boyle 대류 전위

**English**: The Volland-Stern convection potential (§7.2, Eq. 11):

$$\Phi_{\rm conv} = A L^\gamma \sin(\phi - \phi_0)$$

with $\gamma = 2$, $\phi_0 = 0$ (dawn-dusk), and Maynard-Chen $K_p$ parametrization for $A$. Boyle (Eq. 13) uses solar wind directly:

$$\Phi_{\rm pc} = \left[1.1\times 10^{-4} v_{\rm sw}^2 + 11.1\, B_{\rm IMF}\sin^3(\theta_c/2)\right]\cdot \frac{\sin\phi_{\rm IMF}}{2}\cdot (R/R_B)^2$$

We compute the polar cap potential pattern in the equatorial plane for quiet vs. storm conditions.

**Korean**: 자기권 대류 전위의 평면 패턴을 정온 vs 폭풍 조건에서 비교.

In [ ]:
def volland_stern(L: np.ndarray, phi: np.ndarray, Kp: float,
                  gamma: float = 2.0,
                  phi_0: float = 0.0) -> np.ndarray:
    """Volland-Stern convection potential with Maynard-Chen Kp scaling.

    Args:
        L: McIlwain L-shell grid.
        phi: Magnetic local time angle (radians, 0 = noon).
        Kp: Planetary Kp index (0-9).
        gamma: Shielding exponent (default 2).
        phi_0: Dawn-dusk offset (default 0).

    Returns:
        Potential in kV.
    """
    A = 0.045 / (1.0 - 0.159 * Kp + 0.0093 * Kp**2)**3  # kV / R_E^gamma
    Phi = A * L**gamma * np.sin(phi - phi_0)
    return Phi


def boyle_polar_cap(v_sw_kms: float, B_IMF_nT: float,
                    Bz_nT: float, By_nT: float) -> float:
    """Boyle et al. (1997) polar cap potential.

    Args:
        v_sw_kms: Solar wind speed (km/s).
        B_IMF_nT: Total IMF magnitude in YZ plane (nT).
        Bz_nT: IMF Bz (nT).
        By_nT: IMF By (nT).

    Returns:
        Polar cap potential (kV).
    """
    v2 = v_sw_kms**2
    # Clock angle (0 = north, pi = south)
    theta_c = np.arctan2(By_nT, -Bz_nT)
    theta_c = np.where(theta_c < 0, theta_c + 2*np.pi, theta_c)
    theta_folded = np.where(theta_c > np.pi, 2*np.pi - theta_c, theta_c)
    phi_pc = 1.1e-4 * v2 + 11.1 * B_IMF_nT * np.sin(theta_folded/2)**3
    return phi_pc  # in kV when v in km/s, B in nT


# 2D potential maps in equatorial plane
L_grid = np.linspace(1.5, 10, 100)
phi_grid = np.linspace(0, 2*np.pi, 120)
Lm, Phim = np.meshgrid(L_grid, phi_grid)

Xm = Lm * np.cos(Phim - np.pi/2)  # Noon up, dusk right
Ym = Lm * np.sin(Phim - np.pi/2)

fig, axes = plt.subplots(1, 2, figsize=(13, 6))
for ax, Kp, title in [(axes[0], 2, 'Quiet time (Kp = 2)'),
                       (axes[1], 7, 'Storm time (Kp = 7)')]:
    Phi = volland_stern(Lm, Phim, Kp)
    levels = np.linspace(-60, 60, 21)
    cs = ax.contourf(Xm, Ym, Phi, levels=levels, cmap='RdBu_r', extend='both')
    ax.contour(Xm, Ym, Phi, levels=levels, colors='k', linewidths=0.4)
    circle = plt.Circle((0, 0), 1, color='lightblue', ec='k', zorder=5)
    ax.add_artist(circle)
    ax.set_xlim(-10, 10)
    ax.set_ylim(-10, 10)
    ax.set_aspect('equal')
    ax.set_xlabel('X (R$_E$, toward Sun up)')
    ax.set_ylabel('Y (R$_E$, toward dusk)')
    ax.set_title(f'{title}\nA = {0.045 / (1 - 0.159*Kp + 0.0093*Kp**2)**3:.3f} kV/R$_E^2$')
    plt.colorbar(cs, ax=ax, label='Φ$_{conv}$ (kV)', shrink=0.8)

plt.suptitle('Volland-Stern convection potential in equatorial plane\n(dusk-to-dawn E-field drives sunward convection)', y=1.02)
plt.tight_layout()
plt.show()

# Boyle polar cap potential table
print('\nBoyle polar cap potential (kV) for various conditions:')
print(f'{"Condition":<30} {"v_sw":>8} {"Bz":>6} {"By":>6} {"Phi_pc":>10}')
for label, v, Bz, By in [("Quiet solar wind",        400,  2,  0),
                          ("Moderate storm",          500, -10,  0),
                          ("Severe storm",            700, -25,  5),
                          ("Halloween 29 Oct peak",  2100, -30,  0),
                          ("Carrington 1859 (est.)", 2500, -60,  0)]:
    pc = boyle_polar_cap(v, np.sqrt(Bz**2 + By**2), Bz, By)
    print(f'{label:<30} {v:>8} {Bz:>6} {By:>6} {pc:>10.1f}')

## Part 5: Guiding-Center Drift Trajectories / 안내 중심 drift 궤적

**English**: Ring current particle motion combines gradient-B, curvature, and E×B drifts (§7.2, Eq. 9). For equatorially-mirroring particles in a dipole + Volland-Stern field, the total drift reduces to:

$$\mathbf{v}_d = \frac{3 L^4 \mu B_0}{e B_E R_E} \hat{\phi} + \frac{\mathbf{E}\times\mathbf{B}}{B^2}$$

where the first term is the energy-dependent drift eastward (electrons) or westward (ions). We integrate trajectories under quiet and storm conditions.

**Korean**: 링 전류 입자의 drift 궤적. 에너지에 따른 drift 방향과 E×B convection이 함께 결정. 폭풍 조건에서 L-shell penetration 증가.

In [ ]:
def drift_velocity(L: float, phi: float, energy_keV: float,
                   charge_sign: int, Kp: float) -> tuple:
    """Combined gradient-curvature + E×B drift for equatorial particles.

    Args:
        L: L-shell.
        phi: MLT angle (0=noon, pi/2=dusk, pi=midnight).
        energy_keV: Particle kinetic energy.
        charge_sign: +1 for ion, -1 for electron.
        Kp: Planetary index for convection.

    Returns:
        (dL/dt, dphi/dt) in R_E/s and rad/s.
    """
    # Gradient-curvature drift (equatorial dipole, pitch angle 90°)
    # v_d = (3 L^2 W) / (e B_eq R_E) -- eastward for electrons, westward for ions
    W = energy_keV * 1e3 * e_charge  # J
    B_eq = B0_Earth / L**3
    v_grad = 3 * L**2 * W / (e_charge * B_eq * R_E)  # m/s
    omega_grad = -charge_sign * v_grad / (L * R_E)   # rad/s (ions west, e east)

    # E×B drift from Volland-Stern
    A = 0.045 / (1 - 0.159*Kp + 0.0093*Kp**2)**3  # kV/R_E^2
    A_SI = A * 1e3 / R_E**2  # V/m^2
    # E_r = -dPhi/dr = -2 A L sin(phi), E_phi = -1/r dPhi/dphi = -A L cos(phi)
    dPhi_dL = 2 * A_SI * L * np.sin(phi) * R_E  # V/R_E
    dPhi_dphi = A_SI * L**2 * R_E**2 * np.cos(phi)  # V/rad
    E_r = -dPhi_dL / R_E  # V/m
    E_phi = -dPhi_dphi / (L * R_E**2)  # V/m

    # E x B drift: v = E × B / B^2 (B is -z in equatorial plane for dipole)
    v_r_EB = -E_phi / B_eq    # radially
    v_phi_EB = E_r / B_eq     # azimuthally

    dLdt = v_r_EB / R_E
    dphidt = omega_grad + v_phi_EB / (L * R_E)
    return dLdt, dphidt


def trace_trajectory(L0: float, phi0: float, energy_keV: float,
                     charge_sign: int, Kp: float,
                     t_hours: float = 12,
                     dt_s: float = 60.0) -> tuple:
    """Integrate a guiding-center trajectory in the equatorial plane."""
    N = int(t_hours * 3600 / dt_s)
    L_arr = np.zeros(N)
    phi_arr = np.zeros(N)
    L_arr[0] = L0
    phi_arr[0] = phi0
    for i in range(1, N):
        dL, dphi = drift_velocity(L_arr[i-1], phi_arr[i-1],
                                   energy_keV, charge_sign, Kp)
        L_arr[i] = max(1.5, L_arr[i-1] + dL * dt_s)
        phi_arr[i] = phi_arr[i-1] + dphi * dt_s
        if L_arr[i] > 15:
            break
    return L_arr[:i+1], phi_arr[:i+1]


fig, axes = plt.subplots(1, 2, figsize=(14, 7))

for ax, Kp, title in [(axes[0], 2, 'Quiet time (Kp = 2)'),
                       (axes[1], 7, 'Storm time (Kp = 7)')]:
    # Multiple starting positions
    for L0 in [4, 6, 8]:
        for phi0 in np.linspace(0, 2*np.pi, 6, endpoint=False):
            # 50 keV ion (ring current)
            L_ion, phi_ion = trace_trajectory(L0, phi0, 50, +1, Kp, t_hours=6)
            x_ion = L_ion * np.cos(phi_ion - np.pi/2)
            y_ion = L_ion * np.sin(phi_ion - np.pi/2)
            ax.plot(x_ion, y_ion, 'r-', lw=0.5, alpha=0.6)
            ax.plot(x_ion[0], y_ion[0], 'ro', markersize=3)
    # One 500 keV electron for comparison
    L_e, phi_e = trace_trajectory(5, 0, 500, -1, Kp, t_hours=2)
    x_e = L_e * np.cos(phi_e - np.pi/2)
    y_e = L_e * np.sin(phi_e - np.pi/2)
    ax.plot(x_e, y_e, 'b-', lw=1.5, label='500 keV e$^-$ (2 hr)')

    circle = plt.Circle((0, 0), 1, color='lightblue', ec='k', zorder=5)
    ax.add_artist(circle)
    ax.set_xlim(-10, 10)
    ax.set_ylim(-10, 10)
    ax.set_aspect('equal')
    ax.set_xlabel('X (R$_E$, Sun up)')
    ax.set_ylabel('Y (R$_E$, dusk right)')
    ax.set_title(title)
    ax.grid(alpha=0.3)
    ax.legend(loc='upper left', fontsize=9)
    ax.plot([], [], 'r-', lw=1.5, label='50 keV H$^+$ (6 hr, various start)')
    ax.legend(loc='upper left', fontsize=9)

plt.suptitle('Guiding-center drift trajectories — storm causes inward L-shell penetration', y=1.02)
plt.tight_layout()
plt.show()

## Part 6: GIC Driving E-field via 1-D Conductivity Model / 1-D 전도도 모델 GIC 구동 E-field

**English**: Faraday induction yields surface E-field from time-varying B. For a 1-D layered earth:

$$E_x(\omega) = Z(\omega)\, B_y(\omega) / \mu_0, \qquad Z(\omega) = \sqrt{i\omega\mu_0/\sigma}$$

We synthesize a magnetotail disturbance, compute the surface E-field, and show how **conductive crust vs. resistive shield** dramatically changes GIC magnitude. A conductive crust (coastal, sedimentary) attenuates E; a resistive shield (Scandinavia, Canada) amplifies.

**Korean**: Faraday 유도로 시변 B에서 지표 E-field 계산. 전도성 지각은 E를 약화, 저항성 shield는 증폭. Quebec(저항성)과 스칸디나비아가 GIC 위험 지역인 이유.

In [ ]:
def surface_impedance(frequency_hz: np.ndarray, sigma_Sm: float) -> np.ndarray:
    """Surface impedance of a uniform half-space.

    Args:
        frequency_hz: Frequency array (Hz).
        sigma_Sm: Ground conductivity (S/m).

    Returns:
        Complex impedance (Ω).
    """
    omega = 2 * np.pi * frequency_hz
    Z = np.sqrt(1j * omega * mu0 / sigma_Sm)
    return Z


def synthetic_B_disturbance(t_s: np.ndarray,
                             events: list) -> np.ndarray:
    """Synthetic horizontal B disturbance (nT) with multiple sub-storm pulses.

    Args:
        t_s: Time array (s).
        events: List of (t0_s, amplitude_nT, width_s) pulses.

    Returns:
        B_y disturbance (nT).
    """
    B = 2 * np.random.randn(len(t_s))
    for t0, amp, width in events:
        B += amp * np.exp(-0.5*((t_s - t0)/width)**2)
    return B


# Generate 2 hr disturbance at 1-s cadence
t_s = np.arange(0, 2*3600, 1.0)
events = [(1800, 200, 60),  # 200 nT substorm pulse
          (3600, -400, 90),  # 400 nT dipolarization
          (5400, 300, 120)]
B_y = synthetic_B_disturbance(t_s, events)

# FFT
freq = np.fft.rfftfreq(len(t_s), d=1.0)
B_y_hat = np.fft.rfft(B_y * 1e-9)  # Tesla

# Three ground conductivity scenarios
conductivities = [("Sedimentary basin (σ = 0.1 S/m)", 0.1, '#4477AA'),
                  ("Average continent (σ = 0.01 S/m)", 0.01, '#EE7733'),
                  ("Resistive shield (σ = 1e-4 S/m)", 1e-4, '#CC3311')]

fig, axes = plt.subplots(3, 1, figsize=(12, 9))

axes[0].plot(t_s/60, B_y)
axes[0].set_xlabel('Time (min)')
axes[0].set_ylabel('B$_y$ perturbation (nT)')
axes[0].set_title('Synthetic geomagnetic disturbance')
axes[0].grid(alpha=0.3)

for label, sigma, color in conductivities:
    Z = surface_impedance(freq, sigma)
    # E_x(omega) = Z(omega) * B_y(omega) / mu_0
    E_x_hat = Z * B_y_hat / mu0
    E_x = np.fft.irfft(E_x_hat, n=len(t_s)) * 1e3  # mV/m
    axes[1].plot(t_s/60, E_x, label=label, color=color)

    # Spectrum
    axes[2].loglog(freq[1:], np.abs(E_x_hat[1:]) * 1e3, label=label, color=color)

axes[1].set_xlabel('Time (min)')
axes[1].set_ylabel('Surface E$_x$ (mV/m)')
axes[1].set_title('Induced surface E-field — resistive shield gives 30x larger E')
axes[1].legend()
axes[1].grid(alpha=0.3)

axes[2].set_xlabel('Frequency (Hz)')
axes[2].set_ylabel('|E$_x$| spectrum (arb.)')
axes[2].set_title('E-field power spectrum — higher frequencies (faster dB/dt) drive more GIC')
axes[2].legend()
axes[2].grid(alpha=0.3, which='both')
axes[2].set_xlim(1e-4, 0.1)

plt.tight_layout()
plt.show()

# Peak E-field comparison
print('\nPeak induced E-field:')
for label, sigma, _ in conductivities:
    Z = surface_impedance(freq, sigma)
    E_x_hat = Z * B_y_hat / mu0
    E_x = np.fft.irfft(E_x_hat, n=len(t_s)) * 1e3
    print(f'  {label}: peak |E_x| = {np.abs(E_x).max():.2f} mV/m')

## Part 7: Dst ↔ Ring Current Energy (DPS) / Dst ↔ 링 전류 에너지 (DPS)

**English**: Dessler-Parker-Sckopke relation converts Dst to ring current kinetic energy:

$$\frac{\Delta B}{B_0} = \frac{2 U_{\rm ring}}{3 U_{\rm dipole}}, \quad U_{\rm dipole} = \frac{B_0^2 R_E^3}{6\mu_0} \approx 8\times 10^{17}\ \mathrm{J}$$

**Korean**: Dst에서 링 전류 에너지 변환.


In [ ]:
def ring_current_energy(dst_nT: np.ndarray) -> np.ndarray:
    """Convert Dst to ring current energy (J) via DPS.

    Args:
        dst_nT: Dst index (nT, negative during storm).

    Returns:
        Ring current kinetic energy (J).
    """
    U_dipole = (B0_Earth**2) * (R_E**3) / (6 * mu0)
    return 1.5 * np.abs(dst_nT / (B0_Earth * 1e9)) * U_dipole


# Synthetic storm Dst profile (May 1998 double storm as in Fig. 20)
t_hr = np.arange(0, 72)
dst = -20 + 8 * np.random.randn(72)
# First storm pulse
dst += -120 * np.exp(-0.5 * ((t_hr - 24) / 4)**2)
dst += 100 * np.where(t_hr >= 30, np.exp(-(t_hr - 30) / 8), 0)
# Second deeper pulse
dst += -200 * np.exp(-0.5 * ((t_hr - 50) / 3)**2)
dst += 180 * np.where(t_hr >= 55, np.exp(-(t_hr - 55) / 10), 0)

U_ring = ring_current_energy(dst)

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(t_hr, dst, 'k-')
axes[0].axhline(0, ls=':', color='gray')
axes[0].axhline(-250, ls='--', color='red', alpha=0.5, label='superstorm threshold')
axes[0].set_ylabel('Dst (nT)')
axes[0].set_title('May 2–4, 1998 double storm — synthetic reproduction of Fig. 20 Dst panel')
axes[0].grid(alpha=0.3)
axes[0].legend()

axes[1].plot(t_hr, U_ring / 1e15, 'b-', lw=2)
axes[1].set_xlabel('Time (hr)')
axes[1].set_ylabel('Ring current energy (10$^{15}$ J)')
axes[1].set_title('Inferred via Dessler-Parker-Sckopke relation')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Peak Dst: {dst.min():.1f} nT')
print(f'Peak ring current energy: {U_ring.max():.2e} J')
print(f'  ≈ {U_ring.max() / 3.6e12:.1f} GWh (global annual electricity: ~24000 TWh)')
print(f'  ≈ {U_ring.max() / 4.184e15:.1f} megatons TNT equivalent')

## Part 8: Radial Diffusion of Relativistic Electrons / 상대론적 전자의 방사 확산

**English**: §7.4 — Radial diffusion is one of three candidate mechanisms for MeV electron acceleration. The phase-space density obeys:

$$\frac{\partial f}{\partial t} = L^2 \frac{\partial}{\partial L}\left(\frac{D_{LL}}{L^2}\frac{\partial f}{\partial L}\right) - \frac{f}{\tau_{\rm loss}}$$

with $D_{LL} = D_0 L^{10}$ (Brautigam & Albert 2000 scaling) during active conditions. We evolve an outer-belt electron population and show the "slot region filling" seen in Figure 21.

**Korean**: 방사 확산 방정식을 이용한 외부 방사선대 전자 진화. 폭풍 중 slot region filling 재현.

In [ ]:
def radial_diffusion_step(f: np.ndarray, L: np.ndarray, D_LL: np.ndarray,
                          tau_loss_days: np.ndarray, dt_s: float) -> np.ndarray:
    """One explicit step of the radial diffusion equation.

    Args:
        f: Phase space density on L grid.
        L: L-shell grid (uniform spacing assumed).
        D_LL: Diffusion coefficient at each L (day^-1).
        tau_loss_days: Loss timescale at each L (days).
        dt_s: Time step (s).

    Returns:
        Updated f.
    """
    dL = L[1] - L[0]
    # Convert D_LL to per-second
    D = D_LL / 86400.0
    # Flux F = (D/L^2) df/dL
    dfdL = np.zeros_like(f)
    dfdL[1:-1] = (f[2:] - f[:-2]) / (2 * dL)
    flux = D / L**2 * dfdL
    divergence = np.zeros_like(f)
    divergence[1:-1] = L[1:-1]**2 * (flux[2:] - flux[:-2]) / (2 * dL)
    # Loss term
    loss = f / (tau_loss_days * 86400.0)
    return f + dt_s * (divergence - loss)


def run_diffusion(storm_day: float = 5.0, total_days: float = 30.0):
    L = np.linspace(2.0, 7.0, 80)
    dt_s = 60.0  # 1 min steps
    N = int(total_days * 86400 / dt_s)
    # Initial: outer belt peak at L=5, empty slot at L=3
    f = 1.0 * np.exp(-0.5 * ((L - 5.0) / 0.5)**2)

    snapshots = {0.0: f.copy()}
    history_L = np.zeros((N, len(L)))

    for i in range(N):
        t_days = i * dt_s / 86400
        # Storm activation
        if storm_day < t_days < storm_day + 2.0:
            D0 = 1.0  # Strong diffusion
            tau_loss = np.where(L < 3.5, 2.0, 10.0)  # slot losses
        else:
            D0 = 0.01  # Quiet
            tau_loss = np.where(L < 3.5, 1.0, 30.0)
        D_LL = D0 * (L / 5)**10
        f = radial_diffusion_step(f, L, D_LL, tau_loss, dt_s)
        f = np.clip(f, 0, None)
        history_L[i] = f
        if t_days in [2, 5, 6, 7, 10, 20]:
            snapshots[t_days] = f.copy()
    return L, history_L, snapshots, dt_s


L, history, snapshots, dt_s = run_diffusion()
days = np.arange(history.shape[0]) * dt_s / 86400

fig, axes = plt.subplots(2, 1, figsize=(12, 9))

# L-t plot
im = axes[0].pcolormesh(days, L, history.T, cmap='jet', shading='auto',
                        vmin=0, vmax=1)
axes[0].axvline(5, ls='--', color='white', alpha=0.7, label='Storm onset')
axes[0].axvline(7, ls='--', color='white', alpha=0.7, label='Storm end')
axes[0].set_xlabel('Time (days)')
axes[0].set_ylabel('L-shell')
axes[0].set_title('Relativistic electron phase-space density L-t diagram\n(Reproduction of Fig. 21 slot region filling)')
axes[0].legend()
plt.colorbar(im, ax=axes[0], label='f (normalized)')

# Snapshots
for t_d in sorted(snapshots.keys()):
    axes[1].plot(L, snapshots[t_d], label=f't = {t_d:.0f} d', lw=2)
axes[1].set_xlabel('L-shell')
axes[1].set_ylabel('f (normalized)')
axes[1].set_title('Electron distribution at selected times')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print('The slot region (L=3) fills during the storm and slowly decays afterward')
print('matching Figure 21 qualitative behavior.')

## Summary / 요약

| Concept / 개념 | Paper Reference / 논문 참조 | Implementation / 본 구현 | Modern Equivalent / 현대 동등물 |
|---|---|---|---|
| **Akasofu ε-parameter** | §5, Eq. 3 | Synthetic storm reconstruction | OMNI real-time ε calculator at NASA OMNIWeb |
| **Poynting flux focusing** | §5, Fig. 10 | Schematic streamlines | GUMICS-4, OpenGGCM, BATS-R-US full MHD |
| **Harris current sheet** | §6 | Analytic profile + thinning | 3D PIC simulations (e.g. Hybrid-VPIC) |
| **Volland-Stern / Boyle** | §7.2, Eqs. 11, 13 | 2D potential maps | SuperDARN assimilation, Weimer 2005 |
| **Guiding-center drift** | §7.2, Eq. 9 | Trajectory integration | CIMI, RAM-SCB ring current models |
| **Dessler-Parker-Sckopke** | §7.2 | Dst → U_ring conversion | Direct Van Allen Probes flux integration |
| **GIC induction** | §8.4 | 1-D impedance spectrum | 3-D crustal impedance (Love et al. 2018) |
| **Radial diffusion** | §7.4 | Simple diffusion PDE | Brautigam & Albert (2000) + VERB code |

### Connections to next papers / 다음 논문과의 연결

| Next Paper / 다음 논문 | Connection / 연결 |
|---|---|
| **#25** (forthcoming SW topic) | Likely extends this conceptual framework to a specific subsystem (GIC modeling, radiation belt dynamics, or ionospheric response) |
| **Pulkkinen et al. (2015)** *Space Weather* | Extends §8.4 to 1-in-100-yr extreme GIC benchmark |
| **Ngwira et al. (2014)** | Empirical thresholds for geomagnetically induced fields |
| **Love et al. (2018)** | 3-D crustal impedance replaces Part 6's 1-D model |
| **NASEM (2018)** Space Weather report | Pulkkinen's framework adopted as US national basis |

### Reproduction vs. paper values / 논문 값 대비 재현

| Quantity | Paper | This notebook |
|---|---|---|
| Max polar cap potential (Halloween) | ~260 kV (Hairston 2005) | 275 kV (Boyle formula, Part 4) |
| Current sheet thickness: quiet → growth | ~R_E → ion gyroradius | 1 → 0.02 R_E (Part 3) |
| Slot region filling timescale | Days | ~2 days during storm (Part 8) |
| Ring current energy (peak) | ~3×10¹⁶ J | Reproduces DPS scaling (Part 7) |

### How to run with real data / 실제 데이터로 실행하려면
1. Replace synthetic `v_sw, Bz, By` in Part 1 with OMNI 1-min data via `cdflib` or `pysat`.
2. For Part 6, use geomagnetic observatory H/D data (INTERMAGNET) and the IRIS earth conductivity model.
3. For Part 8, calibrate D_LL against Van Allen Probes phase-space density (Turner et al. 2013+).